In [1]:
import random
import io
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import imagehash

import torch
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToPILImage

In [2]:
# --------------------------
# 1) Dataset CIFAR-10
# --------------------------
to_pil = ToPILImage()
ds = CIFAR10(root="./data", train=True, download=True)

def get_image(idx: int) -> Image.Image:
    x, y = ds[idx]
    # x est déjà PIL Image avec torchvision CIFAR10, mais on garde robuste:
    if not isinstance(x, Image.Image):
        x = to_pil(x)
    return x.convert("RGB")

100%|██████████| 170M/170M [33:20<00:00, 85.2kB/s]   


In [3]:
# --------------------------
# 2) Transformations "permises" (bénignes)
# --------------------------
def jpeg_recompress(img: Image.Image, quality: int = 30) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def add_gaussian_noise(img: Image.Image, sigma: float = 8.0) -> Image.Image:
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr, mode="RGB")

def change_brightness(img: Image.Image, factor: float = 1.2) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def change_contrast(img: Image.Image, factor: float = 1.2) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def blur(img: Image.Image, radius: float = 1.0) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def small_rotate(img: Image.Image, degrees: float = 5.0) -> Image.Image:
    # expand=False garde la même taille, mais introduit du remplissage
    return img.rotate(degrees, resample=Image.BILINEAR, expand=False)

BENIGN_TRANSFORMS = [
    lambda im: jpeg_recompress(im, quality=30),
    lambda im: add_gaussian_noise(im, sigma=8.0),
    lambda im: change_brightness(im, factor=1.2),
    lambda im: change_contrast(im, factor=1.2),
    lambda im: blur(im, radius=1.0),
    lambda im: small_rotate(im, degrees=5.0),
]

In [4]:
# --------------------------
# 3) Transformations "non permises" (sémantiques / agressives)
# --------------------------
def big_rotate(img: Image.Image, degrees: float = 30.0) -> Image.Image:
    return img.rotate(degrees, resample=Image.BILINEAR, expand=False)

def big_shift(img: Image.Image, shift: int = 6) -> Image.Image:
    # translation simple (CIFAR 32x32)
    arr = np.array(img)
    out = np.zeros_like(arr)
    out[:, shift:, :] = arr[:, :-shift, :]
    return Image.fromarray(out, mode="RGB")

NON_ALLOWED_TRANSFORMS = [
    lambda im: big_rotate(im, degrees=30.0),
    lambda im: big_shift(im, shift=6),
]

In [5]:
# --------------------------
# 4) Hash functions (images)
#    IMPORTANT: CIFAR est petit (32x32).
#    Pour stabiliser, on upscale avant hash.
# --------------------------
UPSCALE = (128, 128)

def prep(img: Image.Image) -> Image.Image:
    return img.resize(UPSCALE, resample=Image.BILINEAR).convert("RGB")

HASHERS = {
    "aHash": lambda im: imagehash.average_hash(prep(im)),
    "dHash": lambda im: imagehash.dhash(prep(im)),
    "pHash": lambda im: imagehash.phash(prep(im)),
    "wHash": lambda im: imagehash.whash(prep(im)),
}

def hamming(h1, h2) -> int:
    return h1 - h2  # imagehash définit "-" comme distance de Hamming


In [6]:
# --------------------------
# 5) Générer des paires positives/négatives et mesurer
# --------------------------
def sample_pairs(n_pos=1000, n_neg=1000, seed=0):
    random.seed(seed)
    N = len(ds)

    pos_pairs = []
    for _ in range(n_pos):
        i = random.randrange(N)
        img = get_image(i)
        t = random.choice(BENIGN_TRANSFORMS)
        img_t = t(img)
        pos_pairs.append((img, img_t))

    neg_pairs = []
    for _ in range(n_neg):
        i = random.randrange(N)
        j = random.randrange(N)
        while j == i:
            j = random.randrange(N)
        neg_pairs.append((get_image(i), get_image(j)))

    return pos_pairs, neg_pairs

def compute_distances(pairs, hasher):
    dists = []
    for a, b in pairs:
        ha = hasher(a)
        hb = hasher(b)
        dists.append(hamming(ha, hb))
    return np.array(dists, dtype=np.int32)


In [7]:
# 6) Choisir un seuil τ selon un objectif
# --------------------------
def choose_threshold_by_fpr(pos_dists, neg_dists, target_fpr=0.01):
    # τ max tel que FPR <= target_fpr
    # FPR(τ) = P(dist_neg <= τ)
    neg_sorted = np.sort(neg_dists)
    idx = int(np.floor(target_fpr * len(neg_sorted)))
    idx = max(0, min(idx, len(neg_sorted)-1))
    tau = neg_sorted[idx]
    return int(tau)

def evaluate_at_tau(pos_dists, neg_dists, tau):
    tp = int(np.sum(pos_dists <= tau))
    fn = int(np.sum(pos_dists > tau))
    fp = int(np.sum(neg_dists <= tau))
    tn = int(np.sum(neg_dists > tau))

    recall = tp / (tp + fn + 1e-9)
    fpr = fp / (fp + tn + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {"tau": tau, "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "recall": recall, "precision": precision, "fpr": fpr, "f1": f1}


In [8]:
# --------------------------
# 7) Run benchmark
# --------------------------
if __name__ == "__main__":
    pos_pairs, neg_pairs = sample_pairs(n_pos=1500, n_neg=1500, seed=42)

    for name, hasher in HASHERS.items():
        pos_d = compute_distances(pos_pairs, hasher)
        neg_d = compute_distances(neg_pairs, hasher)

        # Objectif: FPR <= 1%
        tau = choose_threshold_by_fpr(pos_d, neg_d, target_fpr=0.01)
        metrics = evaluate_at_tau(pos_d, neg_d, tau)

        print(f"\n=== {name} ===")
        print("pos dist: mean", float(pos_d.mean()), "p95", float(np.percentile(pos_d, 95)))
        print("neg dist: mean", float(neg_d.mean()), "p01", float(np.percentile(neg_d, 1)))
        print("Chosen tau (FPR<=1%):", tau)
        print("Metrics:", metrics)



=== aHash ===
pos dist: mean 1.5433333333333332 p95 7.0
neg dist: mean 31.614 p01 12.99
Chosen tau (FPR<=1%): 13
Metrics: {'tau': 13, 'TP': 1493, 'FP': 22, 'TN': 1478, 'FN': 7, 'recall': 0.9953333333326698, 'precision': 0.985478547854135, 'fpr': 0.01466666666665689, 'f1': 0.990381425701677}

=== dHash ===
pos dist: mean 3.546666666666667 p95 12.0
neg dist: mean 31.828666666666667 p01 20.0
Chosen tau (FPR<=1%): 20
Metrics: {'tau': 20, 'TP': 1499, 'FP': 18, 'TN': 1482, 'FN': 1, 'recall': 0.9993333333326672, 'precision': 0.9881344759387026, 'fpr': 0.011999999999992, 'f1': 0.9937023528304807}

=== pHash ===
pos dist: mean 2.925333333333333 p95 14.0
neg dist: mean 31.508 p01 22.0
Chosen tau (FPR<=1%): 22
Metrics: {'tau': 22, 'TP': 1499, 'FP': 28, 'TN': 1472, 'FN': 1, 'recall': 0.9993333333326672, 'precision': 0.9816633922717868, 'fpr': 0.018666666666654223, 'f1': 0.9904195568168613}

=== wHash ===
pos dist: mean 1.952 p95 10.0
neg dist: mean 31.491333333333333 p01 12.0
Chosen tau (FPR<=1%)